In [1]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from sklearn.cluster import KMeans

In [2]:
random_state = 2025

In [3]:
df_ipipneo_120 = pd.read_csv("../data/raw/df_ipipneo_120_clusters")

In [4]:
df_ipipneo_120

,Unnamed: 0,case,sex,age,sec,min,hour,day,month,year,...,facet_modesty,facet_sympathy,neuroticism,facet_anxiety,facet_anger,facet_depression,facet_self_consciousness,facet_immoderation,facet_vulnerability,clusters
0,0,1,2,19,8,41,16,30,6,101,...,74,77,35,27,11,58,39,96,11,1
1,1,2,2,22,24,45,16,30,6,101,...,69,67,66,58,89,89,57,1,80,2
2,2,6,1,13,14,6,17,30,6,101,...,24,12,22,33,86,29,7,6,15,2
3,3,7,2,18,25,11,17,30,6,101,...,38,41,12,12,57,18,8,44,5,1
4,4,8,2,24,19,25,17,30,6,101,...,50,55,26,21,7,26,38,62,53,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
410163,410163,690852,2,29,35,18,23,15,4,111,...,50,79,58,76,76,68,21,33,53,1
410164,410164,690853,2,16,0,35,23,15,4,111,...,57,66,1,1,6,11,30,24,1,1
410165,410165,690854,2,22,9,52,23,15,4,111,...,86,79,96,95,69,89,99,42,97,3
410166,410166,690858,2,15,31,36,1,16,4,111,...,74,66,81,75,73,92,58,44,76,3


In [11]:
print(*df_ipipneo_120.columns)

Unnamed: 0 case sex age sec min hour day month year country i1 i2 i3 i4 i5 i6 i7 i8 i9 i10 i11 i12 i13 i14 i15 i16 i17 i18 i19 i20 i21 i22 i23 i24 i25 i26 i27 i28 i29 i30 i31 i32 i33 i34 i35 i36 i37 i38 i39 i40 i41 i42 i43 i44 i45 i46 i47 i48 i49 i50 i51 i52 i53 i54 i55 i56 i57 i58 i59 i60 i61 i62 i63 i64 i65 i66 i67 i68 i69 i70 i71 i72 i73 i74 i75 i76 i77 i78 i79 i80 i81 i82 i83 i84 i85 i86 i87 i88 i89 i90 i91 i92 i93 i94 i95 i96 i97 i98 i99 i100 i101 i102 i103 i104 i105 i106 i107 i108 i109 i110 i111 i112 i113 i114 i115 i116 i117 i118 i119 i120 openness facet_imagination facet_artistic_interests facet_emotionality facet_adventurousness facet_intellect facet_liberalism conscientiousness facet_self_efficacy facet_orderliness facet_dutifulness facet_achievement_striving facet_self_discipline facet_cautiousness extraversion facet_friendliness facet_gregariousness facet_assertiveness facet_activity_level facet_excitement_seeking facet_cheerfulness agreeableness facet_trust facet_morality f

In [14]:
# Списки фич
traits = ['openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism']
facets = [col for col in df_ipipneo_120.columns if col.startswith('facet_')]

##  Значения по центроиду

In [9]:
trait_target_values = {
    0: {"openness": 22, "conscientiousness": 99, "extraversion": 79, "agreeableness": 90, "neuroticism": 3},
    1: {"openness": 74, "conscientiousness": 97, "extraversion": 45, "agreeableness": 40, "neuroticism": 6},
    2: {"openness": 40, "conscientiousness": 63, "extraversion": 52, "agreeableness": 57, "neuroticism": 62},
    3: {"openness": 88, "conscientiousness": 31, "extraversion": 30, "agreeableness": 42, "neuroticism": 54},
}

In [10]:
facet_target_values = {
    0: {"facet_anger": 1, "facet_orderliness": 98, "facet_self_efficacy": 98, "facet_imagination": 4, "facet_cheerfulness": 94},
    1: {"facet_immoderation": 1, "facet_trust": 1, "facet_gregariousness": 2, "facet_self_efficacy": 97, "facet_orderliness": 96},
    2: {"facet_cheerfulness": 1, "facet_achievement_striving": 84, "facet_anger": 82, "facet_intellect": 18, "facet_morality": 81},
    3: {"facet_artistic_interests": 94, "facet_dutifulness": 7, "facet_excitement_seeking": 12, "facet_emotionality": 84, "facet_anger": 20},
}

In [ ]:
def mae_vs_target(df, target_dict, features, cl):
    sub = df[df['clusters'] == cl].copy()
    results = {}
    for feat in features:
        if feat in target_dict[cl]:
            target = target_dict[cl][feat]
            mae = np.abs(sub[feat] - target).mean()
            results[feat] = {
                'MAE': round(mae, 2),
            }
    return pd.DataFrame(results).T

print("=== MAE ОТ TARGET-ЗНАЧЕНИЙ ===")
for cl in range(4):
    print(f"\nКластер {cl} ({len(df_ipipneo_120[df_ipipneo_120['clusters']==cl])} человек):")
    print("Big Five:")
    print(mae_vs_target(df_ipipneo_120, trait_target_values, traits, cl))
    print("\nВыбранные facets:")
    print(mae_vs_target(df_ipipneo_120, facet_target_values, facets, cl))

=== MAE ОТ TARGET-ЗНАЧЕНИЙ ===

Кластер 0 (107174 человек):
Big Five:
                     MAE  MAE_%         N
openness           18.74  18.74  107174.0
conscientiousness  28.12  28.12  107174.0
extraversion       42.83  42.83  107174.0
agreeableness      35.35  35.35  107174.0
neuroticism        42.34  42.34  107174.0

Выбранные facets:
                       MAE  MAE_%         N
facet_imagination    25.93  25.93  107174.0
facet_self_efficacy  38.34  38.34  107174.0
facet_orderliness    32.60  32.60  107174.0
facet_cheerfulness   45.71  45.71  107174.0
facet_anger          39.92  39.92  107174.0

Кластер 1 (108880 человек):
Big Five:
                     MAE  MAE_%         N
openness           31.09  31.09  108880.0
conscientiousness  18.59  18.59  108880.0
extraversion       30.82  30.82  108880.0
agreeableness      31.29  31.29  108880.0
neuroticism        13.38  13.38  108880.0

Выбранные facets:
                        MAE  MAE_%         N
facet_self_efficacy   23.04  23.04  1088

In [ ]:
посчитать средние абсолютное октлнение по каждой черет относительно значений центроида по 1 кластеру

In [ ]:
посчитать средние абсолютное октлнение по каждой черет относительно значений центроида по 2 кластеру

In [ ]:
посчитать средние абсолютное октлнение по каждой черет относительно значений центроида по 3 кластеру

## Значения усредненные по кластеру

In [15]:
# ====================== 2. РЕАЛЬНЫЕ СРЕДНИЕ ПО КЛАСТЕРАМ ======================
cluster_means = df_ipipneo_120.groupby('clusters')[traits + facets].mean().round(2)
print("\n=== РЕАЛЬНЫЕ СРЕДНИЕ ЗНАЧЕНИЯ ПО КЛАСТЕРАМ (Big Five) ===")
print(cluster_means[traits])


=== РЕАЛЬНЫЕ СРЕДНИЕ ЗНАЧЕНИЯ ПО КЛАСТЕРАМ (Big Five) ===
          openness  conscientiousness  extraversion  agreeableness  \
clusters                                                             
0            25.71              70.88         36.23          55.24   
1            47.85              78.84         75.02          68.52   
2            45.63              37.18         69.48          38.64   
3            40.39              26.81         22.13          39.93   

          neuroticism  
clusters               
0               45.32  
1               17.36  
2               44.31  
3               78.77  


In [ ]:
def mad_vs_own_mean(df, cluster_means, features, cl):
    sub = df[df['clusters'] == cl].copy()
    mean_row = cluster_means.loc[cl]
    results = {}
    for feat in features:
        mae = np.abs(sub[feat] - mean_row[feat]).mean()
        results[feat] = {'MAD': round(mae, 2)}
    return pd.DataFrame(results).T

print("\n=== ВНУТРИКЛАСТЕРНЫЙ MAD (только кластер 0, как просил) ===")
print(mad_vs_own_mean(df_ipipneo_120, cluster_means, traits + facets[:15], 0))  # Big Five + первые 15 facets


=== ВНУТРИКЛАСТЕРНЫЙ MAD (только кластер 0, как просил) ===
                              MAD  MAD_%
openness                    19.17  19.17
conscientiousness           14.85  14.85
extraversion                15.84  15.84
agreeableness               20.20  20.20
neuroticism                 17.42  17.42
facet_imagination           22.09  22.09
facet_artistic_interests    23.45  23.45
facet_emotionality          23.11  23.11
facet_adventurousness       19.93  19.93
facet_intellect             23.80  23.80
facet_liberalism            23.55  23.55
facet_self_efficacy         18.82  18.82
facet_orderliness           22.43  22.43
facet_dutifulness           17.98  17.98
facet_achievement_striving  19.63  19.63
facet_self_discipline       18.48  18.48
facet_cautiousness          18.06  18.06
facet_friendliness          20.21  20.21
facet_gregariousness        19.85  19.85
facet_assertiveness         22.43  22.43


In [17]:
# Сравнение твоего текущего медоида и нового простого среднего
for cl in range(4):
    print(f"\nКластер {cl}:")
    
    # Big Five
    mae_traits = np.abs(
        pd.Series(trait_target_values[cl]) - 
        cluster_means.loc[cl, traits]
    ).mean()
    
    # Ключевые facets (те, что ты выбрал)
    facet_keys = list(facet_target_values[cl].keys())
    mae_facets = np.abs(
        pd.Series(facet_target_values[cl]) - 
        cluster_means.loc[cl, facet_keys]
    ).mean()
    
    print(f"  Big Five MAE: {mae_traits:.1f} баллов")
    print(f"  Ключевые facets MAE: {mae_facets:.1f} баллов")

=== СРАВНЕНИЕ МЕДОИД vs MEAN (MAE в баллах) ===

Кластер 0:
  Big Five MAE: 30.3 баллов
  Ключевые facets MAE: 36.2 баллов

Кластер 1:
  Big Five MAE: 22.8 баллов
  Ключевые facets MAE: 43.5 баллов

Кластер 2:
  Big Five MAE: 17.0 баллов
  Ключевые facets MAE: 40.9 баллов

Кластер 3:
  Big Five MAE: 17.3 баллов
  Ключевые facets MAE: 37.3 баллов


In [19]:
def get_mae_table(cluster: int) -> pd.DataFrame:
    """
    Возвращает таблицу MAE для указанного кластера:
    строки = черты + ключевые фасеты
    столбцы = MAE_to_medoid | MAE_to_mean | Delta
    """
    if cluster not in range(4):
        raise ValueError("Кластер должен быть 0, 1, 2 или 3")
    
    sub = df_ipipneo_120[df_ipipneo_120['clusters'] == cluster].copy()
    n = len(sub)
    
    rows = []
    
    # === 1. Big Five ===
    for feat in ['openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism']:
        medoid_val = trait_target_values[cluster][feat]
        mean_val   = cluster_means.loc[cluster, feat]
        
        mae_medoid = np.abs(sub[feat] - medoid_val).mean()
        mae_mean   = np.abs(sub[feat] - mean_val).mean()
        delta      = mae_medoid - mae_mean
        
        rows.append({
            'feature': feat,
            'MAE_to_medoid': round(mae_medoid, 2),
            'MAE_to_mean':   round(mae_mean, 2),
            'Delta':         round(delta, 2)
        })
    
    # === 2. Ключевые фасеты для этого кластера ===
    facet_keys = list(facet_target_values[cluster].keys())
    for feat in facet_keys:
        medoid_val = facet_target_values[cluster][feat]
        mean_val   = cluster_means.loc[cluster, feat]
        
        mae_medoid = np.abs(sub[feat] - medoid_val).mean()
        mae_mean   = np.abs(sub[feat] - mean_val).mean()
        delta      = mae_medoid - mae_mean
        
        rows.append({
            'feature': feat,
            'MAE_to_medoid': round(mae_medoid, 2),
            'MAE_to_mean':   round(mae_mean, 2),
            'Delta':         round(delta, 2)
        })
    
    df_table = pd.DataFrame(rows)
    df_table = df_table.set_index('feature')
    
    # Красивый вывод
    print(f"\n=== MAE ТАБЛИЦА ДЛЯ КЛАСТЕРА {cluster} ({n:,} человек) ===")
    print(f"Delta > 0 = переход на среднее УЛУЧШАЕТ стабильность")
    return df_table.style.format("{:.2f}").set_caption(f"Кластер {cluster} — MAE от медиоида vs среднего")

In [21]:
get_mae_table(0)


=== MAE ТАБЛИЦА ДЛЯ КЛАСТЕРА 0 (107,174 человек) ===
Delta > 0 = переход на среднее УЛУЧШАЕТ стабильность


,MAE_to_medoid,MAE_to_mean,Delta
feature,,,
openness,18.74,19.17,-0.43
conscientiousness,28.12,14.85,13.28
extraversion,42.83,15.84,26.98
agreeableness,35.35,20.20,15.15
neuroticism,42.34,17.42,24.93
facet_anger,39.92,22.96,16.96
facet_orderliness,32.60,22.43,10.17
facet_self_efficacy,38.34,18.82,19.53
facet_imagination,25.93,22.09,3.84


In [22]:
get_mae_table(1)


=== MAE ТАБЛИЦА ДЛЯ КЛАСТЕРА 1 (108,880 человек) ===
Delta > 0 = переход на среднее УЛУЧШАЕТ стабильность


,MAE_to_medoid,MAE_to_mean,Delta
feature,,,
openness,31.09,23.02,8.07
conscientiousness,18.59,13.65,4.94
extraversion,30.82,13.22,17.60
agreeableness,31.29,16.81,14.48
neuroticism,13.38,11.92,1.46
facet_immoderation,32.13,20.34,11.79
facet_trust,65.81,18.96,46.86
facet_gregariousness,68.10,18.81,49.28
facet_self_efficacy,23.04,16.59,6.45


In [24]:
get_mae_table(2)


=== MAE ТАБЛИЦА ДЛЯ КЛАСТЕРА 2 (108,867 человек) ===
Delta > 0 = переход на среднее УЛУЧШАЕТ стабильность


,MAE_to_medoid,MAE_to_mean,Delta
feature,,,
openness,24.19,24.02,0.17
conscientiousness,27.94,17.03,10.91
extraversion,20.57,13.91,6.66
agreeableness,25.52,20.50,5.02
neuroticism,23.05,17.55,5.50
facet_cheerfulness,63.78,16.60,47.18
facet_achievement_striving,41.23,21.84,19.39
facet_anger,32.18,24.01,8.17
facet_intellect,31.12,25.33,5.78


In [25]:
get_mae_table(3)


=== MAE ТАБЛИЦА ДЛЯ КЛАСТЕРА 3 (85,247 человек) ===
Delta > 0 = переход на среднее УЛУЧШАЕТ стабильность


,MAE_to_medoid,MAE_to_mean,Delta
feature,,,
openness,48.27,23.91,24.36
conscientiousness,17.49,16.89,0.61
extraversion,16.76,15.00,1.76
agreeableness,22.75,22.61,0.14
neuroticism,26.75,13.70,13.06
facet_artistic_interests,51.40,25.20,26.20
facet_dutifulness,30.36,22.05,8.31
facet_excitement_seeking,30.59,23.00,7.59
facet_emotionality,37.80,24.58,13.22
